# 투수 임베딩 v1 — 시즌 순방향 OOF 피처

팀의 다음 단계 모델이 학습 데이터에서 투수 임베딩을 안전하게 사용할 수 있도록 시즌 순방향 OOF(out-of-fold) lookup을 만듭니다.

- 시즌 `s`의 임베딩 모델은 `season < s`인 정답만 학습합니다.
- 시즌 `s`의 모든 행에는 시즌 첫 투구 이전에 확정된 동일한 투수-시즌 임베딩을 붙입니다.
- 출력은 `(pitcher_id, season)`당 한 행입니다.
- 48차원 계약: 투수 ID 16 + 과거 Trackman tower 24 + 신인/경험 cohort 8
- 2019~2020은 충분한 이전 Trackman-supervised 모델을 만들 수 없어 0 벡터와 `oof_available=False`를 제공합니다.

> component 모델은 reverse/middle 보조 라벨을 사용하므로 운영진 답변 전까지 실험 피처입니다. 테스트 행이나 테스트 분포는 전혀 사용하지 않습니다.


In [1]:
from pathlib import Path
import gc
import json
import math
import random
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import sparse
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 빠른 검증이 끝난 뒤 False로 바꾸면 전체 행을 사용합니다.
QUICK_RUN = True
MAX_TRAIN_ROWS = 350_000 if QUICK_RUN else None
MAX_VALID_ROWS = 100_000 if QUICK_RUN else None
EPOCHS = 3 if QUICK_RUN else 12
BATCH_SIZE = 4096

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "../../data/train.csv").resolve().exists():
    # 프로젝트 루트에서 직접 실행한 경우
    PROJECT_ROOT = Path.cwd()
else:
    PROJECT_ROOT = (NOTEBOOK_DIR / "../..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "experiment" / "pitcher_embedding" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"project={PROJECT_ROOT}")
print(f"device={DEVICE}, torch={torch.__version__}, quick_run={QUICK_RUN}")


project=C:\Users\isj67\Desktop\LGAIMERS
device=cuda, torch=2.5.1+cu121, quick_run=True


## 1. 데이터 로드

Trackman은 ID 연결과 과거 구위 요약에 필요한 열만 읽습니다. 메인 데이터는 모델 후보 피처와 보조 라벨 복원 열을 읽습니다.

In [2]:
ASOF_COLS = [
    "asof_pitcher_n", "asof_pitcher_success_rate", "asof_pitcher_reverse_rate",
    "asof_pitcher_middle_rate", "asof_pitcher_ball_rate", "asof_pitcher_strike_rate",
    "asof_pitcher_prev1_game_success_rate", "asof_pitcher_prev3_game_success_rate",
    "asof_pitcher_prev5_game_success_rate", "asof_pitcher_prev1_game_middle_rate",
    "asof_pitcher_prev3_game_middle_rate", "asof_pitcher_prev5_game_middle_rate",
    "asof_batter_n", "asof_batter_success_rate", "asof_batter_middle_rate",
    "asof_pitcher_pitchmix_n", "asof_pitcher_fastball_rate",
    "asof_pitcher_breaking_rate", "asof_pitcher_offspeed_rate",
]

CONTEXT_COLS = [
    "row_id", "season", "game_month", "game_dayofweek", "inning", "top_bottom", "game_type",
    "balls_before", "strikes_before", "outs_before", "run_top_before", "run_bot_before",
    "run_total_before", "score_diff_home", "score_diff_pitcher_team",
    "runner_on_1b", "runner_on_2b", "runner_on_3b", "num_runners_on", "base_state",
    "home_win_expectancy", "away_win_expectancy", "li", "pitcher_id", "batter_id",
    "pitcher_hand", "batter_hand", "pitcher_team_id", "batter_team_id",
]

TM_FINGERPRINT_COLS = [
    "season", "pitcher_trackman_id", "pitcher_hand", "pitcher_team", "game_month",
    "game_dayofweek", "inning", "top_bottom", "balls_before", "strikes_before",
    "outs_before", "batter_hand",
]
TM_METRICS = [
    "rel_speed", "spin_rate", "induced_vert_break", "horz_break",
    "extension", "rel_height", "rel_side", "zone_speed",
]
TM_COLS = list(dict.fromkeys(TM_FINGERPRINT_COLS + ["pitch_type_group"] + TM_METRICS))

train = pd.read_csv(DATA_DIR / "train.csv", usecols=CONTEXT_COLS + ASOF_COLS + ["control_success"])
tm = pd.read_csv(DATA_DIR / "trackman_history.csv", usecols=TM_COLS)

print("train:", train.shape, "pitchers:", train.pitcher_id.nunique())
print("trackman:", tm.shape, "pitchers:", tm.pitcher_trackman_id.nunique())
display(train.head(3))


train: (1475092, 49) pitchers: 792
trackman: (1793078, 21) pitchers: 906


,row_id,season,game_month,game_dayofweek,inning,top_bottom,game_type,balls_before,strikes_before,outs_before,run_top_before,run_bot_before,run_total_before,score_diff_home,score_diff_pitcher_team,runner_on_1b,runner_on_2b,runner_on_3b,num_runners_on,base_state,home_win_expectancy,away_win_expectancy,li,pitcher_id,batter_id,pitcher_hand,batter_hand,pitcher_team_id,batter_team_id,asof_pitcher_n,asof_pitcher_success_rate,asof_pitcher_reverse_rate,asof_pitcher_middle_rate,asof_pitcher_ball_rate,asof_pitcher_strike_rate,asof_pitcher_prev1_game_success_rate,asof_pitcher_prev3_game_success_rate,asof_pitcher_prev5_game_success_rate,asof_pitcher_prev1_game_middle_rate,asof_pitcher_prev3_game_middle_rate,asof_pitcher_prev5_game_middle_rate,asof_batter_n,asof_batter_success_rate,asof_batter_middle_rate,asof_pitcher_pitchmix_n,asof_pitcher_fastball_rate,asof_pitcher_breaking_rate,asof_pitcher_offspeed_rate,control_success
0,TRAIN_0000001,2019,3,5,1,T,R,0,0,0,0,0,0,0,0,0,0,0,0,___,50.0,50.0,0.87,21961,21996,1,2,16,13,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN,NaN,NaN,0
1,TRAIN_0000002,2019,3,5,1,T,R,0,0,0,0,0,0,0,0,1,0,0,1,1__,46.5,53.5,1.44,21961,22103,1,1,16,13,1,0.0,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,1,1.0,0.0,0.0,0
2,TRAIN_0000003,2019,3,5,1,T,R,1,0,0,0,0,0,0,0,1,0,0,1,1__,46.5,53.5,1.44,21961,22103,1,1,16,13,2,0.0,0.5,0.0,0.5,0.0,NaN,NaN,NaN,NaN,NaN,NaN,1,0.0,0.0,2,1.0,0.0,0.0,0


## 2. 세부 실패 라벨 복원

같은 투수의 다음 투구 직전 누적값은 현재 투구까지 반영한 누적값입니다. 따라서

`현재 사건 = (다음 누적 건수 × 다음 누적률) - (현재 누적 건수 × 현재 누적률)`

로 현재 투구의 사건 여부를 복원할 수 있습니다. 마지막 투구처럼 다음 누적값이 없는 행은 학습에서 제외합니다.

In [3]:
def recover_binary_increment(df, rate_col):
    # 같은 투수의 다음 as-of 누적값을 이용해 현재 행의 0/1 사건을 복원한다.
    n = df["asof_pitcher_n"].astype(float)
    cumulative = n * df[rate_col].fillna(0.0).astype(float)
    next_n = df.groupby("pitcher_id", sort=False)["asof_pitcher_n"].shift(-1)
    next_cumulative = cumulative.groupby(df["pitcher_id"], sort=False).shift(-1)
    delta = next_cumulative - cumulative
    rounded = np.rint(delta)
    # 저장된 누적률은 소수점 반올림값이라 원래 0/1 사건에서 약 0.015까지 흔들린다.
    # 정수 반올림 결과가 0/1이고 반올림 오차가 0.05 미만일 때만 사용한다.
    valid = next_n.eq(n + 1) & rounded.isin([0, 1]) & (delta - rounded).abs().lt(0.05)
    recovered = pd.Series(np.nan, index=df.index, dtype="float32")
    recovered.loc[valid] = rounded.loc[valid].astype("float32")
    return recovered, valid


train["y_success_recovered"], valid_success = recover_binary_increment(train, "asof_pitcher_success_rate")
train["y_reverse"], valid_reverse = recover_binary_increment(train, "asof_pitcher_reverse_rate")
train["y_middle"], valid_middle = recover_binary_increment(train, "asof_pitcher_middle_rate")
train["y_ball_result"], valid_ball = recover_binary_increment(train, "asof_pitcher_ball_rate")

train["component_label_valid"] = valid_success & valid_reverse & valid_middle
train["y_far_residual"] = (
    train["control_success"].eq(0)
    & train["y_reverse"].eq(0)
    & train["y_middle"].eq(0)
).astype("float32")

valid = train["component_label_valid"]
reconstruction_accuracy = (
    train.loc[valid, "y_success_recovered"].astype(int)
    == train.loc[valid, "control_success"].astype(int)
).mean()

failure = train.loc[valid & train.control_success.eq(0)].copy()
failure["case"] = np.select(
    [
        failure.y_reverse.eq(1) & failure.y_middle.eq(0),
        failure.y_reverse.eq(0) & failure.y_middle.eq(1),
        failure.y_reverse.eq(1) & failure.y_middle.eq(1),
    ],
    ["reverse only", "middle only", "reverse & middle"],
    default="far residual",
)

label_report = pd.DataFrame({
    "value": [
        int(valid.sum()),
        reconstruction_accuracy,
        train.loc[valid, "y_reverse"].mean(),
        train.loc[valid, "y_middle"].mean(),
        train.loc[valid, "y_far_residual"].mean(),
        int((valid & train.control_success.eq(1) & train.y_reverse.eq(1)).sum()),
        int((valid & train.control_success.eq(1) & train.y_middle.eq(1)).sum()),
    ]
}, index=[
    "valid rows", "success reconstruction accuracy", "reverse rate", "middle rate",
    "far residual rate", "success & reverse contradictions", "success & middle contradictions",
])

display(label_report)
display((failure["case"].value_counts().to_frame("n").assign(rate=lambda x: x.n / x.n.sum())).sort_index())


,value
valid rows,1.474300e+06
success reconstruction accuracy,1.000000e+00
reverse rate,2.290355e-01
middle rate,1.496052e-01
far residual rate,1.317208e-01
success & reverse contradictions,0.000000e+00
success & middle contradictions,0.000000e+00


,n,rate
case,,
far residual,194196,0.276569
middle only,170297,0.242533
reverse & middle,50266,0.071588
reverse only,287401,0.409310


## 3. Main ↔ Trackman 투수 ID crosswalk

직접 공통 ID가 없으므로 같은 시즌의 투구 상황 분포를 투수별 지문으로 사용합니다.

- 월, 요일, 이닝, 초/말, 볼·스트라이크·아웃, 상대 타자 손잡이의 결합 빈도
- 같은 시즌·같은 투수 손잡이 후보끼리 cosine similarity 비교
- 최고 유사도 `≥ 0.80`, 2위와의 차이 `≥ 0.02`만 고신뢰 매칭으로 채택
- 여러 시즌에서 고신뢰 매칭된 경우 동일 Trackman ID인지 검사
- 하나의 Trackman ID에 복수 Main ID가 걸리면 평균 신뢰도가 높은 하나만 남김

이 연결은 타깃을 전혀 사용하지 않습니다.

In [4]:
def make_state_code(df, is_main):
    month = df["game_month"].to_numpy(np.int64)
    dow = df["game_dayofweek"].to_numpy(np.int64)
    inning = np.minimum(df["inning"].to_numpy(np.int64), 20)
    if is_main:
        bottom = df["top_bottom"].astype(str).eq("B").to_numpy(np.int64)
        batter_right = df["batter_hand"].to_numpy(np.int64) == 2
    else:
        bottom = df["top_bottom"].astype(str).eq("Bottom").to_numpy(np.int64)
        batter_right = df["batter_hand"].astype(str).eq("Right").to_numpy(np.int64)

    code = month.copy()
    for values, base in [
        (dow, 7), (inning, 21), (bottom, 2),
        (df["balls_before"].to_numpy(np.int64), 4),
        (df["strikes_before"].to_numpy(np.int64), 3),
        (df["outs_before"].to_numpy(np.int64), 3),
        (batter_right, 2),
    ]:
        code = code * base + values
    return code


def build_pitcher_season_matches(main_df, tm_df):
    main_fp = main_df[[
        "season", "pitcher_id", "pitcher_hand", "game_month", "game_dayofweek", "inning",
        "top_bottom", "balls_before", "strikes_before", "outs_before", "batter_hand",
    ]].copy()
    tm_fp = tm_df[TM_FINGERPRINT_COLS].copy()
    main_fp["state"] = make_state_code(main_fp, is_main=True)
    tm_fp["state"] = make_state_code(tm_fp, is_main=False)

    records = []
    for season in sorted(main_fp.season.unique()):
        a = main_fp[main_fp.season.eq(season)]
        b = tm_fp[tm_fp.season.eq(season)]
        main_ids = np.sort(a.pitcher_id.unique())
        tm_ids = np.sort(b.pitcher_trackman_id.unique())
        main_index = {v: i for i, v in enumerate(main_ids)}
        tm_index = {v: i for i, v in enumerate(tm_ids)}
        states = np.union1d(a.state.unique(), b.state.unique())
        state_index = {v: i for i, v in enumerate(states)}

        ag = a.groupby(["pitcher_id", "state"], sort=False).size().reset_index(name="n")
        bg = b.groupby(["pitcher_trackman_id", "state"], sort=False).size().reset_index(name="n")
        A = sparse.csr_matrix(
            (ag.n, (ag.pitcher_id.map(main_index), ag.state.map(state_index))),
            shape=(len(main_ids), len(states)), dtype=np.float32,
        )
        B = sparse.csr_matrix(
            (bg.n, (bg.pitcher_trackman_id.map(tm_index), bg.state.map(state_index))),
            shape=(len(tm_ids), len(states)), dtype=np.float32,
        )
        A = sparse.diags(1 / np.sqrt(A.multiply(A).sum(1).A1).clip(1e-9)) @ A
        B = sparse.diags(1 / np.sqrt(B.multiply(B).sum(1).A1).clip(1e-9)) @ B
        similarity = (A @ B.T).toarray()

        main_hand = a.groupby("pitcher_id").pitcher_hand.first().reindex(main_ids).to_numpy()
        tm_hand_raw = b.groupby("pitcher_trackman_id").pitcher_hand.first().reindex(tm_ids).to_numpy()
        tm_hand = np.where(tm_hand_raw == "Left", 1, 2)
        similarity[main_hand[:, None] != tm_hand[None, :]] = -1

        order = np.argsort(similarity, axis=1)
        best, second = order[:, -1], order[:, -2]
        main_n = a.groupby("pitcher_id").size().reindex(main_ids).to_numpy()
        tm_n = b.groupby("pitcher_trackman_id").size().reindex(tm_ids).to_numpy()

        for i, pitcher_id in enumerate(main_ids):
            records.append({
                "season": season,
                "pitcher_id": pitcher_id,
                "pitcher_trackman_id": tm_ids[best[i]],
                "similarity": float(np.clip(similarity[i, best[i]], -1, 1)),
                "second_similarity": float(np.clip(similarity[i, second[i]], -1, 1)),
                "margin": float(similarity[i, best[i]] - similarity[i, second[i]]),
                "main_n": int(main_n[i]),
                "trackman_n": int(tm_n[best[i]]),
            })
    return pd.DataFrame(records)


pitcher_season_matches = build_pitcher_season_matches(train, tm)
high_conf = pitcher_season_matches.query("similarity >= 0.80 and margin >= 0.02").copy()

vote_table = (
    high_conf.groupby(["pitcher_id", "pitcher_trackman_id"], as_index=False)
    .agg(cw_match_seasons=("season", "nunique"), cw_mean_sim=("similarity", "mean"),
         cw_min_margin=("margin", "min"), cw_total_main_n=("main_n", "sum"))
    .sort_values(["pitcher_id", "cw_match_seasons", "cw_mean_sim", "cw_total_main_n"], ascending=[True, False, False, False])
)
crosswalk = vote_table.drop_duplicates("pitcher_id", keep="first").copy()

# 한 Trackman ID를 여러 Main ID에 붙이지 않는다.
crosswalk = (
    crosswalk.sort_values(["pitcher_trackman_id", "cw_match_seasons", "cw_mean_sim", "cw_total_main_n"],
                          ascending=[True, False, False, False])
    .drop_duplicates("pitcher_trackman_id", keep="first")
    .sort_values("pitcher_id")
    .reset_index(drop=True)
)

check = high_conf.merge(crosswalk[["pitcher_id", "pitcher_trackman_id"]], on="pitcher_id", suffixes=("", "_selected"))
season_agreement = (check.pitcher_trackman_id == check.pitcher_trackman_id_selected).mean()
row_coverage = train.pitcher_id.isin(crosswalk.pitcher_id).mean()

crosswalk.to_parquet(OUTPUT_DIR / "main_trackman_pitcher_crosswalk.parquet", index=False)
pitcher_season_matches.to_parquet(OUTPUT_DIR / "pitcher_season_match_diagnostics.parquet", index=False)

crosswalk_report = pd.DataFrame({
    "value": [len(pitcher_season_matches), len(high_conf), len(crosswalk), season_agreement, row_coverage]
}, index=["all pitcher-seasons", "high-confidence pitcher-seasons", "accepted pitchers", "multi-season agreement", "main row coverage"])
display(crosswalk_report)
display(pitcher_season_matches[["similarity", "margin"]].describe(percentiles=[.1, .25, .5, .75, .9, .95, .99]))
display(crosswalk.head())


,value
all pitcher-seasons,2260.000000
high-confidence pitcher-seasons,1019.000000
accepted pitchers,419.000000
multi-season agreement,1.000000
main row coverage,0.862979


,similarity,margin
count,2260.000000,2260.000000
mean,0.682311,0.580588
std,0.265606,0.268299
min,0.004991,0.000026
10%,0.286582,0.163440
25%,0.469816,0.366351
50%,0.751401,0.658948
75%,0.926175,0.820061
90%,0.964597,0.861767
95%,0.981711,0.882933


,pitcher_id,pitcher_trackman_id,cw_match_seasons,cw_mean_sim,cw_min_margin,cw_total_main_n
0,20700,99445,2,0.925940,0.854215,959
1,20938,70425,1,0.888533,0.810863,638
2,21170,71837,1,1.000000,0.904556,20
3,21279,72447,2,0.872431,0.802840,802
4,21372,72523,4,0.936453,0.841218,3602


## 4. 이전 완료 시즌 Trackman 피처와 신인 cohort

예측 시즌 `s`의 입력에는 `season < s`인 Trackman 행만 사용합니다. 구속·회전·무브먼트의 평균/표준편차, 구종군 비율, 이전 시즌 및 누적 투구 수를 만듭니다.

신인/저표본 cohort는 다음처럼 모델 입력 시점에 알 수 있는 값만 사용합니다.

- `UNSEEN`: 현재까지 메인 이력도 없고 과거 Trackman도 없음
- `ROOKIE_1_25`: 과거 어떤 완료 시즌도 100구를 넘지 않았고 현재 누적 1~25구
- `ROOKIE_26_100`: 같은 조건에서 현재 누적 26~100구
- `RETURNING`: 과거 100구 초과 시즌은 있으나 직전 시즌 Trackman 기록 없음
- `VETERAN`: 그 외

`same_season_low_volume`은 요청하신 “그 시즌 100구 이하” 분석 태그지만 시즌 종료 후에만 확정되므로 모델 입력에서 제외합니다.

In [5]:
def build_lagged_trackman(tm_df, cutoffs=range(2019, 2026)):
    frames = []
    season_counts = (
        tm_df.groupby(["pitcher_trackman_id", "season"]).size()
        .rename("season_n").reset_index()
    )

    for cutoff in cutoffs:
        prior = tm_df[tm_df.season.lt(cutoff)]
        if prior.empty:
            continue
        grouped = prior.groupby("pitcher_trackman_id", sort=False)
        stats = grouped[TM_METRICS].agg(["mean", "std"])
        stats.columns = [f"tm_{metric}_{stat}" for metric, stat in stats.columns]
        stats = stats.reset_index()
        stats["tm_prior_n"] = grouped.size().reindex(stats.pitcher_trackman_id).to_numpy()

        mix = pd.crosstab(prior.pitcher_trackman_id, prior.pitch_type_group, normalize="index")
        for group_name in ["fastball", "breaking", "offspeed", "other"]:
            if group_name not in mix.columns:
                mix[group_name] = 0.0
        mix = mix[["fastball", "breaking", "offspeed", "other"]]
        mix.columns = [f"tm_pitch_group_{c}_rate" for c in mix.columns]
        mix = mix.reset_index()

        prior_counts = season_counts[season_counts.season.lt(cutoff)]
        max_n = prior_counts.groupby("pitcher_trackman_id").season_n.max().rename("tm_prior_max_season_n")
        prev_n = (
            season_counts[season_counts.season.eq(cutoff - 1)]
            .set_index("pitcher_trackman_id").season_n.rename("tm_prev_season_n")
        )

        out = stats.merge(mix, on="pitcher_trackman_id", how="left")
        out = out.join(max_n, on="pitcher_trackman_id").join(prev_n, on="pitcher_trackman_id")
        out["season"] = cutoff
        frames.append(out)
    return pd.concat(frames, ignore_index=True)


lagged_tm = build_lagged_trackman(tm)
model_df = train.merge(crosswalk, on="pitcher_id", how="left")
model_df = model_df.merge(lagged_tm, on=["pitcher_trackman_id", "season"], how="left")

for col in ["tm_prior_n", "tm_prev_season_n", "tm_prior_max_season_n"]:
    model_df[col] = model_df[col].fillna(0)
model_df["tm_available"] = model_df.tm_prior_n.gt(0).astype("float32")

established = model_df.tm_prior_max_season_n.gt(100)
current_n = model_df.asof_pitcher_n
model_df["experience_cohort"] = np.select(
    [
        current_n.eq(0) & model_df.tm_prior_n.eq(0),
        ~established & current_n.le(25),
        ~established & current_n.le(100),
        established & model_df.tm_prev_season_n.eq(0),
    ],
    ["UNSEEN", "ROOKIE_1_25", "ROOKIE_26_100", "RETURNING"],
    default="VETERAN",
)

season_volume = model_df.groupby(["season", "pitcher_id"]).size().rename("same_season_pitch_n")
model_df = model_df.join(season_volume, on=["season", "pitcher_id"])
model_df["same_season_low_volume"] = model_df.same_season_pitch_n.le(100)

feature_coverage = (
    model_df.groupby("season")
    .agg(rows=("row_id", "size"), crosswalk_rate=("pitcher_trackman_id", lambda s: s.notna().mean()),
         prior_trackman_rate=("tm_available", "mean"), low_volume_rate=("same_season_low_volume", "mean"))
)
display(feature_coverage)
display(pd.crosstab(model_df.season, model_df.experience_cohort, normalize="index").round(4))

del lagged_tm
gc.collect()


,rows,crosswalk_rate,prior_trackman_rate,low_volume_rate
season,,,,
2019,237413,0.870677,0.000000,0.019300
2020,244087,0.877847,0.696153,0.014310
2021,247088,0.862539,0.708885,0.018580
2022,247472,0.873545,0.752065,0.018851
2023,245525,0.867101,0.768089,0.019363
2024,253507,0.827575,0.690734,0.018043


experience_cohort,RETURNING,ROOKIE_1_25,ROOKIE_26_100,UNSEEN,VETERAN
season,,,,,
2019,0.0000,0.0364,0.0924,0.0015,0.8697
2020,0.0000,0.0111,0.0305,0.0004,0.9580
2021,0.0077,0.0093,0.0229,0.0004,0.9597
2022,0.0483,0.0084,0.0208,0.0003,0.9222
2023,0.0209,0.0064,0.0175,0.0003,0.9549
2024,0.0272,0.0077,0.0224,0.0003,0.9424


20

## 5. OOF 학습 설정

2021, 2022, 2023, 2024 시즌을 각각 독립 fold로 만듭니다. 빠른 협업용 생성에서는 fold별 최대 30만 행과 2 epoch를 사용합니다. 모델 성능 제출이 아니라 누수 없는 표현 생성이 목적입니다.

In [6]:
OOF_MAX_ROWS = 300_000
OOF_EPOCHS = 2

HISTORY_FEATURES = [
    "season", "game_month", "game_dayofweek", "inning", "balls_before", "strikes_before", "outs_before",
    "run_top_before", "run_bot_before", "run_total_before", "score_diff_home", "score_diff_pitcher_team",
    "runner_on_1b", "runner_on_2b", "runner_on_3b", "num_runners_on",
    "home_win_expectancy", "away_win_expectancy", "li", "pitcher_hand", "batter_hand",
] + ASOF_COLS
TRACKMAN_COUNT_RAW = ["tm_prior_n", "tm_prev_season_n", "tm_prior_max_season_n"]
TRACKMAN_BASE_FEATURES = [
    c for c in model_df.columns if c.startswith("tm_") and c != "tm_available"
] + ["tm_available"]

for source in ["asof_pitcher_n", "asof_batter_n", "asof_pitcher_pitchmix_n"]:
    new_col = f"log1p_{source}"
    model_df[new_col] = np.log1p(model_df[source].clip(lower=0))
    HISTORY_FEATURES.remove(source)
    HISTORY_FEATURES.append(new_col)
TRACKMAN_FEATURES = TRACKMAN_BASE_FEATURES.copy()
for source in TRACKMAN_COUNT_RAW:
    new_col = f"log1p_{source}"
    model_df[new_col] = np.log1p(model_df[source].clip(lower=0))
    TRACKMAN_FEATURES.remove(source)
    TRACKMAN_FEATURES.append(new_col)

COHORTS = ["UNSEEN", "ROOKIE_1_25", "ROOKIE_26_100", "RETURNING", "VETERAN"]
cohort_to_index = {name: i for i, name in enumerate(COHORTS)}
TARGET_COLS = ["y_reverse", "y_middle", "y_far_residual", "control_success"]

def sample_indices(pool, limit, seed):
    pool = np.asarray(pool)
    if len(pool) <= limit:
        return np.sort(pool)
    return np.sort(np.random.default_rng(seed).choice(pool, limit, replace=False))

def fit_preprocessor(frame, columns):
    x = frame[columns].to_numpy(dtype=np.float64)
    median = np.nanmedian(x, axis=0)
    median = np.where(np.isfinite(median), median, 0.0)
    x = np.where(np.isfinite(x), x, median)
    mean = x.mean(axis=0)
    scale = x.std(axis=0)
    scale = np.where(scale > 1e-8, scale, 1.0)
    return {"median": median.astype("float32"), "mean": mean.astype("float32"), "scale": scale.astype("float32")}

def transform_numeric(frame, columns, prep):
    x = frame[columns].to_numpy(dtype=np.float32)
    x = np.where(np.isfinite(x), x, prep["median"])
    return ((x - prep["mean"]) / prep["scale"]).astype("float32")


In [7]:
class OOFEmbeddingNet(nn.Module):
    def __init__(self, hist_dim, tm_dim, n_pitchers, n_cohorts):
        super().__init__()
        self.pitcher_embedding = nn.Embedding(n_pitchers + 1, 16, padding_idx=0)
        self.cohort_embedding = nn.Embedding(n_cohorts, 8)
        self.history_tower = nn.Sequential(
            nn.Linear(hist_dim, 96), nn.LayerNorm(96), nn.SiLU(), nn.Dropout(0.10),
            nn.Linear(96, 48), nn.SiLU(),
        )
        self.trackman_tower = nn.Sequential(
            nn.Linear(tm_dim, 64), nn.LayerNorm(64), nn.SiLU(), nn.Dropout(0.10),
            nn.Linear(64, 24), nn.SiLU(),
        )
        self.individual_projection = nn.Linear(16, 16)
        self.cohort_projection = nn.Linear(8, 16)
        self.fusion = nn.Sequential(
            nn.Linear(48 + 24 + 16 + 8, 64), nn.SiLU(), nn.Dropout(0.10),
            nn.Linear(64, 32), nn.LayerNorm(32), nn.SiLU(),
        )
        self.reverse_head = nn.Linear(32, 1)
        self.middle_head = nn.Linear(32, 1)
        self.far_head = nn.Linear(32, 1)

    def forward(self, history_x, trackman_x, pitcher_idx, cohort_idx, asof_n):
        history_h = self.history_tower(history_x)
        trackman_h = self.trackman_tower(trackman_x)
        individual = self.individual_projection(self.pitcher_embedding(pitcher_idx))
        cohort_raw = self.cohort_embedding(cohort_idx)
        cohort = self.cohort_projection(cohort_raw)
        alpha = (asof_n / (asof_n + 100.0)).clamp(0, 1).unsqueeze(1)
        alpha = alpha * pitcher_idx.ne(0).float().unsqueeze(1)
        pitcher_h = alpha * individual + (1 - alpha) * cohort
        embedding = self.fusion(torch.cat([history_h, trackman_h, pitcher_h, cohort_raw], dim=1))
        logits = torch.cat([self.reverse_head(embedding), self.middle_head(embedding), self.far_head(embedding)], dim=1)
        return logits

    def static_embedding(self, trackman_x, pitcher_idx, cohort_idx):
        individual = self.individual_projection(self.pitcher_embedding(pitcher_idx))
        trackman = self.trackman_tower(trackman_x)
        cohort = self.cohort_embedding(cohort_idx)
        return torch.cat([individual, trackman, cohort], dim=1)


bce = nn.BCEWithLogitsLoss(reduction="none")

def component_loss(logits, y):
    component_p = torch.sigmoid(logits)
    success_p = torch.prod(1 - component_p, dim=1)
    brier = torch.mean((success_p - y[:, 3]) ** 2)
    reverse = bce(logits[:, 0], y[:, 0]).mean()
    no_reverse = y[:, 0].eq(0)
    middle = bce(logits[no_reverse, 1], y[no_reverse, 1]).mean()
    neither = no_reverse & y[:, 1].eq(0)
    far = bce(logits[neither, 2], y[neither, 2]).mean()
    return brier + 0.10 * (reverse + middle + far), brier


## 6. 시즌 순방향 학습과 48차원 추출

임베딩 입력에서 경기 상황 tower는 제거하고, 투수 ID·과거 Trackman·신인 cohort 표현만 연결합니다. 따라서 같은 시즌의 모든 투구에 동일하게 붙일 수 있습니다.

In [8]:
season_start = (
    model_df.sort_values(["season", "pitcher_id", "asof_pitcher_n", "row_id"])
    .groupby(["season", "pitcher_id"], sort=False, as_index=False)
    .head(1).copy()
)

embedding_columns = [
    *[f"pitcher_embedding_{i:02d}" for i in range(16)],
    *[f"trackman_embedding_{i:02d}" for i in range(24)],
    *[f"cohort_embedding_{i:02d}" for i in range(8)],
]
oof_frames = []

# 2019~2020은 prior supervised Trackman tower가 불충분하므로 명시적 0 fallback.
fallback = season_start[season_start.season.le(2020)][[
    "pitcher_id", "season", "experience_cohort", "asof_pitcher_n",
    "tm_prior_n", "tm_prev_season_n", "tm_available",
]].copy()
fallback["oof_available"] = False
fallback["trained_through_season"] = fallback.season - 1
fallback["pitcher_known_before_season"] = False
for column in embedding_columns:
    fallback[column] = 0.0
oof_frames.append(fallback)

for target_season in range(2021, 2025):
    pool = model_df.index[model_df.component_label_valid & model_df.season.lt(target_season)]
    train_idx = sample_indices(pool, OOF_MAX_ROWS, SEED + target_season)
    train_frame = model_df.loc[train_idx]
    hist_prep = fit_preprocessor(train_frame, HISTORY_FEATURES)
    tm_prep = fit_preprocessor(train_frame, TRACKMAN_FEATURES)
    known_pitchers = np.sort(model_df.loc[model_df.season.lt(target_season), "pitcher_id"].unique())
    pitcher_map = {int(pid): i + 1 for i, pid in enumerate(known_pitchers)}

    history_x = transform_numeric(train_frame, HISTORY_FEATURES, hist_prep)
    trackman_x = transform_numeric(train_frame, TRACKMAN_FEATURES, tm_prep)
    pitcher_idx = train_frame.pitcher_id.map(pitcher_map).fillna(0).astype("int64").to_numpy()
    cohort_idx = train_frame.experience_cohort.map(cohort_to_index).astype("int64").to_numpy()
    asof_n = train_frame.asof_pitcher_n.astype("float32").to_numpy()
    y = train_frame[TARGET_COLS].to_numpy("float32")
    loader = DataLoader(TensorDataset(
        torch.from_numpy(history_x), torch.from_numpy(trackman_x), torch.from_numpy(pitcher_idx),
        torch.from_numpy(cohort_idx), torch.from_numpy(asof_n), torch.from_numpy(y),
    ), batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())

    torch.manual_seed(SEED + target_season)
    fold_model = OOFEmbeddingNet(len(HISTORY_FEATURES), len(TRACKMAN_FEATURES), len(pitcher_map), len(COHORTS)).to(DEVICE)
    optimizer = torch.optim.AdamW(fold_model.parameters(), lr=1e-3, weight_decay=1e-4)
    for epoch in range(1, OOF_EPOCHS + 1):
        fold_model.train()
        total_brier = seen = 0.0
        for hx, tx, pi, ci, nn_, yy in loader:
            hx, tx, pi, ci, nn_, yy = hx.to(DEVICE), tx.to(DEVICE), pi.to(DEVICE), ci.to(DEVICE), nn_.to(DEVICE), yy.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = fold_model(hx, tx, pi, ci, nn_)
            loss, brier_part = component_loss(logits, yy)
            loss.backward()
            nn.utils.clip_grad_norm_(fold_model.parameters(), 5.0)
            optimizer.step()
            total_brier += brier_part.item() * len(yy)
            seen += len(yy)
        print(f"target={target_season} epoch={epoch} train_rows={len(train_idx):,} brier={total_brier/seen:.6f}")

    target = season_start[season_start.season.eq(target_season)].copy()
    target_tm = transform_numeric(target, TRACKMAN_FEATURES, tm_prep)
    target_p = target.pitcher_id.map(pitcher_map).fillna(0).astype("int64").to_numpy()
    target_c = target.experience_cohort.map(cohort_to_index).astype("int64").to_numpy()
    fold_model.eval()
    with torch.inference_mode():
        vector = fold_model.static_embedding(
            torch.from_numpy(target_tm).to(DEVICE), torch.from_numpy(target_p).to(DEVICE),
            torch.from_numpy(target_c).to(DEVICE),
        ).cpu().numpy()

    out = target[[
        "pitcher_id", "season", "experience_cohort", "asof_pitcher_n",
        "tm_prior_n", "tm_prev_season_n", "tm_available",
    ]].reset_index(drop=True)
    out["oof_available"] = True
    out["trained_through_season"] = target_season - 1
    out = pd.concat([out, pd.DataFrame(vector, columns=embedding_columns)], axis=1)
    out["pitcher_known_before_season"] = target_p != 0
    oof_frames.append(out)

oof = pd.concat(oof_frames, ignore_index=True).sort_values(["season", "pitcher_id"]).reset_index(drop=True)
if oof[["season", "pitcher_id"]].duplicated().any():
    raise ValueError("Duplicate pitcher-season key")
if not np.isfinite(oof[embedding_columns].to_numpy()).all():
    raise ValueError("Non-finite OOF embedding")
if (oof.loc[oof.oof_available, "trained_through_season"] >= oof.loc[oof.oof_available, "season"]).any():
    raise ValueError("Temporal leakage detected")

oof_path = OUTPUT_DIR / "pitcher_season_embedding_oof.parquet"
oof.to_parquet(oof_path, index=False)
oof.to_csv(OUTPUT_DIR / "pitcher_season_embedding_oof.csv", index=False)
print("saved:", oof_path)
print("shape:", oof.shape, "available:", oof.oof_available.mean())
display(oof.groupby("season").agg(pitchers=("pitcher_id", "size"), available=("oof_available", "mean"),
                                   known=("pitcher_known_before_season", "mean")))
display(oof.head())


target=2021 epoch=1 train_rows=300,000 brier=0.274790


target=2021 epoch=2 train_rows=300,000 brier=0.246152


target=2022 epoch=1 train_rows=300,000 brier=0.280793


target=2022 epoch=2 train_rows=300,000 brier=0.246521


target=2023 epoch=1 train_rows=300,000 brier=0.275388


target=2023 epoch=2 train_rows=300,000 brier=0.246564


target=2024 epoch=1 train_rows=300,000 brier=0.262779


target=2024 epoch=2 train_rows=300,000 brier=0.246942
saved: C:\Users\isj67\Desktop\LGAIMERS\experiment\pitcher_embedding\outputs\pitcher_season_embedding_oof.parquet
shape: (2260, 58) available: 0.6853982300884955


,pitchers,available,known
season,,,
2019,355,0.0,0.000000
2020,356,0.0,0.000000
2021,386,1.0,0.753886
2022,390,1.0,0.774359
2023,382,1.0,0.835079
2024,391,1.0,0.792839


,pitcher_id,season,experience_cohort,asof_pitcher_n,tm_prior_n,tm_prev_season_n,tm_available,oof_available,trained_through_season,pitcher_known_before_season,pitcher_embedding_00,pitcher_embedding_01,pitcher_embedding_02,pitcher_embedding_03,pitcher_embedding_04,pitcher_embedding_05,pitcher_embedding_06,pitcher_embedding_07,pitcher_embedding_08,pitcher_embedding_09,pitcher_embedding_10,pitcher_embedding_11,pitcher_embedding_12,pitcher_embedding_13,pitcher_embedding_14,pitcher_embedding_15,trackman_embedding_00,trackman_embedding_01,trackman_embedding_02,trackman_embedding_03,trackman_embedding_04,trackman_embedding_05,trackman_embedding_06,trackman_embedding_07,trackman_embedding_08,trackman_embedding_09,trackman_embedding_10,trackman_embedding_11,trackman_embedding_12,trackman_embedding_13,trackman_embedding_14,trackman_embedding_15,trackman_embedding_16,trackman_embedding_17,trackman_embedding_18,trackman_embedding_19,trackman_embedding_20,trackman_embedding_21,trackman_embedding_22,trackman_embedding_23,cohort_embedding_00,cohort_embedding_01,cohort_embedding_02,cohort_embedding_03,cohort_embedding_04,cohort_embedding_05,cohort_embedding_06,cohort_embedding_07
0,20700,2019,UNSEEN,0,0.0,0.0,0.0,False,2018,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,20938,2019,UNSEEN,0,0.0,0.0,0.0,False,2018,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,21095,2019,UNSEEN,0,0.0,0.0,0.0,False,2018,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,21178,2019,UNSEEN,0,0.0,0.0,0.0,False,2018,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,21279,2019,UNSEEN,0,0.0,0.0,0.0,False,2018,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 7. 팀 사용 계약

학습 데이터에는 `pitcher_season_embedding_oof.parquet`를 `(pitcher_id, season)`으로 결합합니다. 2025 평가 행에는 `pitcher_embedding_lookup_2025.parquet`를 `pitcher_id`로 결합합니다.

- OOF 파일: 2019~2024 학습용
- 2025 lookup: 평가/추론용
- 2019~2020 OOF: 0 벡터, `oof_available=False`
- 새 투수: `pitcher_known_before_season=False`, cohort 표현과 Trackman fallback 사용
- reverse/middle 보조 라벨이 불허되면 이 OOF 파일은 폐기하고 direct-success 또는 비지도 임베딩으로 다시 생성해야 합니다.
